# 03 Ridge Regression - House Price Prediction

This is a complete, standalone pipeline for training a Ridge Regression model (L2 Regularization) on dirty house price data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

sns.set(style='whitegrid')

### 1. Data Cleaning & Preprocessing

In [ ]:
def clean_house_data(df):
    df = df.copy().dropna(subset=['Price']).drop_duplicates()
    def parse_lot(v):
        if pd.isna(v): return 10000
        v = str(v).lower()
        num = float(re.findall(r'\d+\.\d+|\d+', v)[0])
        if 'ac' in v: return num * 43560
        return num
    df['Lot_Size'] = df['Lot_Size'].apply(parse_lot)
    df['Bedrooms'] = df['Bedrooms'].apply(lambda x: sum([int(i) for i in str(x).split('+')]) if '+' in str(x) else int(re.sub(r'\D', '', str(x))))
    df['Price'] = df['Price'].clip(upper=df['Price'].quantile(0.99))
    return df

df = clean_house_data(pd.read_csv('../_data/house_prices.csv'))
X_train, X_test, y_train, y_test = train_test_split(df.drop('Price', axis=1), df['Price'], test_size=0.2, random_state=42)

### 2. Pipeline Building & Training

In [ ]:
numeric_features = ['Living_Area', 'Lot_Size', 'Bedrooms', 'Bathrooms', 'Garage_Capacity']
categorical_features = ['Neighborhood', 'Property_Type']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', Ridge(alpha=1.0))])
pipeline.fit(X_train, y_train)
print("Ridge model trained.")

### 3. Evaluation & Visualization

In [ ]:
y_pred = pipeline.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

plt.figure(figsize=(10, 6))
sns.residplot(x=y_test, y=y_pred, color='magenta')
plt.title('Residuals Plot - Ridge Regression')
plt.show()

### 4. Model Saving

In [ ]:
joblib.dump(pipeline, '../_model/house_prediction_ridge.joblib')
print("Model saved.")